In [1]:
# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [2]:
!hdfs dfs -mkdir -p /user/dimassaputra/ecommerce/raw
!hdfs dfs -mkdir -p /user/dimassaputra/ecommerce/processed
!hdfs dfs -ls /user/dimassaputra/ecommerce

Found 2 items
drwxr-xr-x   - dimassaputra supergroup          0 2026-09-07 22:54 /user/dimassaputra/ecommerce/processed
drwxr-xr-x   - dimassaputra supergroup          0 2026-09-07 22:54 /user/dimassaputra/ecommerce/raw


In [3]:
!hdfs dfs -put transaksi_magelang.csv /user/dimassaputra/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/dimassaputra/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/dimassaputra/ecommerce/raw/

# Verifikasi berkas lengkap beserta ukurannya
!hdfs dfs -ls -h /user/dimassaputra/ecommerce/raw

Found 3 items
-rw-r--r--   1 dimassaputra supergroup     12.0 K 2026-09-07 22:55 /user/dimassaputra/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 dimassaputra supergroup     11.9 K 2026-09-07 22:55 /user/dimassaputra/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 dimassaputra supergroup     12.4 K 2026-09-07 22:55 /user/dimassaputra/ecommerce/raw/transaksi_yogyakarta.csv


In [4]:
import io
import subprocess
import pandas as pd

def baca_hdfs_csv(hdfs_path):
    cmd = f"hdfs dfs -cat {hdfs_path}"
    output = subprocess.check_output(cmd, shell=True).decode('utf-8')
    return pd.read_csv(io.StringIO(output))

# Membaca ketiga file dari HDFS
df_magelang = baca_hdfs_csv("/user/dimassaputra/ecommerce/raw/transaksi_magelang.csv")
df_yogya = baca_hdfs_csv("/user/dimassaputra/ecommerce/raw/transaksi_yogyakarta.csv")
df_semarang = baca_hdfs_csv("/user/dimassaputra/ecommerce/raw/transaksi_semarang.csv")

# Gabungkan menjadi satu DataFrame
df_gabungan = pd.concat([df_magelang, df_yogya, df_semarang], ignore_index=True)

# Buktikan ketiga kota ada di dalam data
df_gabungan["kota"].value_counts()

kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64

In [5]:
# 1. Tambah kolom total_pendapatan
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]

# 2. Tabel ringkasan per kota dan per kategori
ringkasan = df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"].sum().reset_index()

# 3. Simpan ke lokal sementara
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

# 4. Upload ke HDFS direktori processed
!hdfs dfs -put -f data_gabungan_bersih.csv /user/dimassaputra/ecommerce/processed/
!hdfs dfs -put -f ringkasan_kota_kategori.csv /user/dimassaputra/ecommerce/processed/

# Verifikasi
!hdfs dfs -ls -h /user/dimassaputra/ecommerce/processed

Found 2 items
-rw-r--r--   1 dimassaputra supergroup     40.3 K 2026-09-07 22:56 /user/dimassaputra/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 dimassaputra supergroup        530 2026-09-07 22:56 /user/dimassaputra/ecommerce/processed/ringkasan_kota_kategori.csv
